In [3]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

from xgboost import XGBRegressor

import matplotlib.pyplot as plt

import shap

Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)


In [4]:
df = pd.read_csv(
    "Reduced_Metallophilic_Dataset.csv"
)

print(df.shape)

df["HX_nearest"] = (
    df["HX_nearest"]
    .fillna(0)
)
hammett_map = {

    "NH2": -0.66,
    "OH" : -0.37,
    "CH3": -0.17,

    "H"  : 0.00,

    "F"  : 0.06,

    "CF3": 0.54,
    "NO2": 0.78
}

df["Hammett_sigma"] = (
    df["R"]
    .map(hammett_map)
)

(144, 36)


In [5]:
cu_sets = {

    "Set_1": [1,7,13,19,25,31,37,43,49,55],

    "Set_2": [2,8,14,20,26,32,38,44,50,56],

    "Set_3": [3,9,15,21,27,33,39,45,51,57],

    "Set_4": [4,10,16,22,28,34,40,46,52,58],

    "Set_5": [5,11,17,23,29,35,41,47,53,59],

    "Set_6": [6,12,18,24,30,36,42,48,54,60]
}

In [6]:
xgb_params = {

    "n_estimators": 200,

    "learning_rate": 0.05,

    "max_depth": 4,

    "random_state": 42
}

In [7]:
# =========================================================
# TARGET
# =========================================================

target_column = 'E'

y = df[target_column]

print(y.shape)
# =========================================================
# QTAIM DESCRIPTORS
# =========================================================

qtaim_columns = [

    'rho',
    'laplacian',
    'V',
    'G',
    'V_over_G'

]


# =========================================================
# GEOMETRY DESCRIPTORS
# =========================================================

geometry_columns = [

    'd',
    'Shortest_M_N',
    'Monomer_Slippage',
    'HX_nearest'

]


# =========================================================
# QM DESCRIPTORS
# =========================================================

qm_columns = (

    qtaim_columns +

    geometry_columns

)


# =========================================================
# ML DESCRIPTORS
# =========================================================

ml_columns = [

    'Metal_atomic_number',
    'Metal_atomic_radius',
    'Metal_electronegativity',
    'Metal_polarizability',
    'Metal_ionization_energy',

    'X_atomic_number',
    'X_atomic_radius',
    'X_electronegativity',
    'X_polarizability',
    'X_ionization_energy',
    
    'nAromAtom',
    'nN',
    'nO',
    'nF',
    'nAtom',
    'Hammett_sigma'

]

# =========================================================
# FULL DESCRIPTOR SET
# =========================================================

full_columns = ( qm_columns + ml_columns )

(144,)


In [8]:
descriptor_sets = {

    "QM":
        qm_columns,

    "ML":
        ml_columns,

    "FULL":
        full_columns
}

In [9]:
def prepare_features(
    train_df,
    test_df,
    columns
):

    scaler = StandardScaler()

    X_train = pd.DataFrame(

        scaler.fit_transform(
            train_df[columns]
        ),

        columns=columns,
        index=train_df.index
    )

    X_test = pd.DataFrame(

        scaler.transform(
            test_df[columns]
        ),

        columns=columns,
        index=test_df.index
    )

    return (
        X_train,
        X_test
    )

In [10]:
# =========================================================
# VALIDATION FUNCTION
# =========================================================

def run_validation(
    df,
    columns,
    test_set,
    xgb_params
):

    # -----------------------------------------------------
    # DEFINE TEST DIMERS
    # -----------------------------------------------------

    test_dimers = [

        f"Cu_{i}"

        for i in test_set

    ]

    # -----------------------------------------------------
    # TRAIN / TEST SPLIT
    # -----------------------------------------------------

    test_df = df[
        df["dimer"].isin(
            test_dimers
        )
    ].copy()

    train_df = df[
        ~df["dimer"].isin(
            test_dimers
        )
    ].copy()

    # -----------------------------------------------------
    # FEATURES
    # -----------------------------------------------------

    X_train, X_test = prepare_features(

        train_df,
        test_df,
        columns

    )

    # -----------------------------------------------------
    # TARGET
    # -----------------------------------------------------

    y_train = train_df["E"]

    y_test = test_df["E"]

    # -----------------------------------------------------
    # MODEL
    # -----------------------------------------------------

    model = XGBRegressor(

        **xgb_params

    )

    # -----------------------------------------------------
    # TRAIN MODEL
    # -----------------------------------------------------

    model.fit(

        X_train,
        y_train

    )

    # -----------------------------------------------------
    # PREDICTIONS
    # -----------------------------------------------------

    y_pred = model.predict(

        X_test

    )

    # -----------------------------------------------------
    # METRICS
    # -----------------------------------------------------

    mae = mean_absolute_error(

        y_test,
        y_pred

    )

    rmse = np.sqrt(

        mean_squared_error(

            y_test,
            y_pred

        )

    )

    r2 = r2_score(

        y_test,
        y_pred

    )

    # -----------------------------------------------------
    # PREDICTION TABLE
    # -----------------------------------------------------

    prediction_df = pd.DataFrame({

        "Dimer":
            test_df["dimer"].values,

        "Actual_E":
            y_test.values,

        "Predicted_E":
            y_pred

    })

    prediction_df["Error"] = (

        prediction_df["Actual_E"]

        -

        prediction_df["Predicted_E"]

    )

    prediction_df["Abs_Error"] = (

        prediction_df["Error"]

        .abs()

    )

    # -----------------------------------------------------
    # RETURN RESULTS
    # -----------------------------------------------------

    return {

        "MAE":
            mae,

        "RMSE":
            rmse,

        "R2":
            r2,

        "Train_Size":
            len(train_df),

        "Test_Size":
            len(test_df),

        "Model":
            model,

        "Predictions":
            prediction_df

    }

In [11]:
test_result = run_validation(

    df,

    ml_columns,

    cu_sets["Set_1"],

    xgb_params

)

print(

    test_result["MAE"],
    test_result["RMSE"],
    test_result["R2"]

)

test_result["Predictions"].head()

0.20391717720031774 0.28956571913768087 0.9957459002689131


,Dimer,Actual_E,Predicted_E,Error,Abs_Error
0,Cu_1,-6.64,-7.381104,0.741104,0.741104
1,Cu_7,-8.70,-8.932101,0.232101,0.232101
2,Cu_13,-9.13,-9.139941,0.009941,0.009941
3,Cu_19,-10.32,-10.148756,-0.171244,0.171244
4,Cu_25,-6.28,-6.281965,0.001965,0.001965


In [12]:
# =========================================================
# RUN ALL VALIDATION TESTS
# =========================================================

summary_rows = []

all_prediction_tables = {}

for descriptor_name, columns in descriptor_sets.items():

    descriptor_predictions = []

    print("\n")
    print("=" * 60)
    print(f"{descriptor_name} VALIDATION")
    print("=" * 60)

    for set_name, test_set in cu_sets.items():

        results = run_validation(

            df=df,

            columns=columns,

            test_set=test_set,

            xgb_params=xgb_params

        )

        # ---------------------------------------------
        # SUMMARY TABLE
        # ---------------------------------------------

        summary_rows.append({

            "Descriptor_Set":
                descriptor_name,

            "Test_Set":
                set_name,

            "Train_Size":
                results["Train_Size"],

            "Test_Size":
                results["Test_Size"],

            "MAE":
                results["MAE"],

            "RMSE":
                results["RMSE"],

            "R2":
                results["R2"]

        })

        # ---------------------------------------------
        # PREDICTION TABLE
        # ---------------------------------------------

        prediction_df = (

            results["Predictions"]

            .copy()

        )

        prediction_df["Descriptor_Set"] = (
            descriptor_name
        )

        prediction_df["Test_Set"] = (
            set_name
        )

        descriptor_predictions.append(
            prediction_df
        )

        # ---------------------------------------------
        # PRINT RESULTS
        # ---------------------------------------------

        print(
            f"{set_name:6s}"
            f" | MAE = {results['MAE']:.4f}"
            f" | RMSE = {results['RMSE']:.4f}"
            f" | R² = {results['R2']:.4f}"
        )

    # -------------------------------------------------
    # COMBINE ALL SIX SETS
    # -------------------------------------------------

    all_prediction_tables[
        descriptor_name
    ] = pd.concat(

        descriptor_predictions,

        ignore_index=True

    )



QM VALIDATION
Set_1  | MAE = 0.8172 | RMSE = 1.1644 | R² = 0.9312
Set_2  | MAE = 0.6582 | RMSE = 0.7350 | R² = 0.9615
Set_3  | MAE = 0.6415 | RMSE = 0.7303 | R² = 0.9687
Set_4  | MAE = 0.5468 | RMSE = 0.5984 | R² = 0.9785
Set_5  | MAE = 1.1694 | RMSE = 1.8740 | R² = 0.8190
Set_6  | MAE = 0.7856 | RMSE = 1.5415 | R² = 0.8205


ML VALIDATION
Set_1  | MAE = 0.2039 | RMSE = 0.2896 | R² = 0.9957
Set_2  | MAE = 0.2314 | RMSE = 0.3352 | R² = 0.9920
Set_3  | MAE = 0.1686 | RMSE = 0.2140 | R² = 0.9973
Set_4  | MAE = 0.1716 | RMSE = 0.2222 | R² = 0.9970
Set_5  | MAE = 0.2223 | RMSE = 0.2856 | R² = 0.9958
Set_6  | MAE = 0.2465 | RMSE = 0.3393 | R² = 0.9913


FULL VALIDATION
Set_1  | MAE = 0.6982 | RMSE = 1.1054 | R² = 0.9380
Set_2  | MAE = 0.3046 | RMSE = 0.4391 | R² = 0.9863
Set_3  | MAE = 0.3186 | RMSE = 0.3837 | R² = 0.9914
Set_4  | MAE = 0.2331 | RMSE = 0.2910 | R² = 0.9949
Set_5  | MAE = 1.0336 | RMSE = 1.8941 | R² = 0.8151
Set_6  | MAE = 0.2021 | RMSE = 0.2619 | R² = 0.9948


In [13]:
# =========================================================
# VALIDATION SUMMARY
# =========================================================

validation_summary = pd.DataFrame(
    summary_rows
)

validation_summary

,Descriptor_Set,Test_Set,Train_Size,Test_Size,MAE,RMSE,R2
0,QM,Set_1,134,10,0.817183,1.164395,0.931212
1,QM,Set_2,134,10,0.658237,0.735030,0.961548
2,QM,Set_3,134,10,0.641503,0.730283,0.968727
3,QM,Set_4,134,10,0.546792,0.598415,0.978456
4,QM,Set_5,134,10,1.169416,1.873992,0.819034
5,QM,Set_6,134,10,0.785594,1.541524,0.820512
6,ML,Set_1,134,10,0.203917,0.289566,0.995746
7,ML,Set_2,134,10,0.231391,0.335237,0.992001
8,ML,Set_3,134,10,0.168643,0.213966,0.997315
9,ML,Set_4,134,10,0.171596,0.222211,0.997029


In [14]:
validation_summary.groupby(
    "Descriptor_Set"
)[
    ["MAE", "RMSE", "R2"]
].mean()

,MAE,RMSE,R2
Descriptor_Set,,,
FULL,0.465045,0.729204,0.953416
ML,0.207397,0.280983,0.994865
QM,0.769788,1.107273,0.913248


In [15]:
validation_summary

all_prediction_tables["QM"]

,Dimer,Actual_E,Predicted_E,Error,Abs_Error,Descriptor_Set,Test_Set
0,Cu_1,-6.64,-8.113832,1.473832,1.473832,QM,Set_1
1,Cu_7,-8.70,-8.808965,0.108965,0.108965,QM,Set_1
2,Cu_13,-9.13,-9.174754,0.044754,0.044754,QM,Set_1
3,Cu_19,-10.32,-8.530622,-1.789378,1.789378,QM,Set_1
4,Cu_25,-6.28,-8.924335,2.644335,2.644335,QM,Set_1
5,Cu_31,-17.17,-16.282015,-0.887985,0.887985,QM,Set_1
6,Cu_37,-15.90,-16.307186,0.407186,0.407186,QM,Set_1
7,Cu_43,-17.41,-17.116142,-0.293858,0.293858,QM,Set_1
8,Cu_49,-17.14,-17.423744,0.283744,0.283744,QM,Set_1
9,Cu_55,-16.31,-16.072203,-0.237797,0.237797,QM,Set_1


In [16]:
all_prediction_tables["ML"]



,Dimer,Actual_E,Predicted_E,Error,Abs_Error,Descriptor_Set,Test_Set
0,Cu_1,-6.64,-7.381104,0.741104,0.741104,ML,Set_1
1,Cu_7,-8.70,-8.932101,0.232101,0.232101,ML,Set_1
2,Cu_13,-9.13,-9.139941,0.009941,0.009941,ML,Set_1
3,Cu_19,-10.32,-10.148756,-0.171244,0.171244,ML,Set_1
4,Cu_25,-6.28,-6.281965,0.001965,0.001965,ML,Set_1
5,Cu_31,-17.17,-16.819092,-0.350908,0.350908,ML,Set_1
6,Cu_37,-15.90,-16.079601,0.179601,0.179601,ML,Set_1
7,Cu_43,-17.41,-17.290630,-0.119370,0.119370,ML,Set_1
8,Cu_49,-17.14,-17.091272,-0.048728,0.048728,ML,Set_1
9,Cu_55,-16.31,-16.494209,0.184209,0.184209,ML,Set_1


In [17]:
all_prediction_tables["FULL"]

,Dimer,Actual_E,Predicted_E,Error,Abs_Error,Descriptor_Set,Test_Set
0,Cu_1,-6.64,-8.162428,1.522428,1.522428,FULL,Set_1
1,Cu_7,-8.70,-8.688719,-0.011281,0.011281,FULL,Set_1
2,Cu_13,-9.13,-9.130925,0.000925,0.000925,FULL,Set_1
3,Cu_19,-10.32,-8.806396,-1.513604,1.513604,FULL,Set_1
4,Cu_25,-6.28,-8.959465,2.679465,2.679465,FULL,Set_1
5,Cu_31,-17.17,-16.687786,-0.482214,0.482214,FULL,Set_1
6,Cu_37,-15.90,-16.219679,0.319679,0.319679,FULL,Set_1
7,Cu_43,-17.41,-17.258877,-0.151123,0.151123,FULL,Set_1
8,Cu_49,-17.14,-17.106043,-0.033957,0.033957,FULL,Set_1
9,Cu_55,-16.31,-16.042290,-0.267710,0.267710,FULL,Set_1


In [18]:
# =========================================================
# SAVE VALIDATION SUMMARY
# =========================================================

validation_summary.to_csv(
    "Validation_Summary.csv",
    index=False
)

print(
    "Saved: Validation_Summary.csv"
)

PermissionError: [Errno 13] Permission denied: 'Validation_Summary.csv'

In [ ]:
# =========================================================
# SAVE QM PREDICTIONS
# =========================================================

all_prediction_tables["QM"].to_csv(
    "QM_All_Predictions.csv",
    index=False
)

print(
    "Saved: QM_All_Predictions.csv"
)

In [ ]:
# =========================================================
# SAVE ML PREDICTIONS
# =========================================================

all_prediction_tables["ML"].to_csv(
    "ML_All_Predictions.csv",
    index=False
)

print(
    "Saved: ML_All_Predictions.csv"
)

In [ ]:
# =========================================================
# SAVE FULL PREDICTIONS
# =========================================================

all_prediction_tables["FULL"].to_csv(
    "FULL_All_Predictions.csv",
    index=False
)

print(
    "Saved: FULL_All_Predictions.csv"
)

In [ ]:
# =========================================================
# ACTUAL vs PREDICTED PLOT
# =========================================================

def plot_actual_vs_predicted(
    prediction_df,
    descriptor_name
):

    plt.figure(
        figsize=(7,7)
    )

    colors = {

        "Set_1": "red",
        "Set_2": "blue",
        "Set_3": "green",
        "Set_4": "orange",
        "Set_5": "purple",
        "Set_6": "brown"
    }

    for set_name, color in colors.items():

        subset = prediction_df[

            prediction_df["Test_Set"]

            ==

            set_name

        ]

        plt.scatter(

            subset["Actual_E"],

            subset["Predicted_E"],

            label=set_name,

            color=color,

            s=60

        )

    # --------------------------------------------
    # OVERALL METRICS
    # --------------------------------------------

    actual = prediction_df["Actual_E"]

    predicted = prediction_df["Predicted_E"]

    r2 = r2_score(
        actual,
        predicted
    )

    mae = mean_absolute_error(
        actual,
        predicted
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual,
            predicted
        )
    )

    min_val = min(
        actual.min(),
        predicted.min()
    )

    max_val = max(
        actual.max(),
        predicted.max()
    )

    plt.plot(

        [min_val,max_val],

        [min_val,max_val],

        "k--",

        linewidth=2

    )

    plt.xlabel(
        "Actual Interaction Energy"
    )

    plt.ylabel(
        "Predicted Interaction Energy"
    )

    plt.title(
        f"{descriptor_name} Validation"
    )

    plt.text(

        0.05,

        0.95,

        f"R² = {r2:.4f}\n"
        f"MAE = {mae:.3f}\n"
        f"RMSE = {rmse:.3f}",

        transform=plt.gca().transAxes,

        verticalalignment="top"

    )

    plt.legend()

    plt.tight_layout()

    plt.show()

In [ ]:
# =========================================================
# GENERATE VALIDATION PLOTS
# =========================================================

for descriptor_name in descriptor_sets.keys():

    plot_actual_vs_predicted(

        all_prediction_tables[
            descriptor_name
        ],

        descriptor_name

    )

In [ ]:
# =========================================================
# QM vs ML vs FULL
# =========================================================

plt.figure(
    figsize=(8,8)
)

descriptor_colors = {

    "QM"  : "red",

    "ML"  : "blue",

    "FULL": "green"
}

for descriptor_name, color in descriptor_colors.items():

    prediction_df = all_prediction_tables[
        descriptor_name
    ]

    plt.scatter(

        prediction_df["Actual_E"],

        prediction_df["Predicted_E"],

        color=color,

        label=descriptor_name,

        s=70,

        alpha=0.8,

        edgecolors="black"

    )

all_actual = pd.concat([

    all_prediction_tables["QM"]["Actual_E"],

    all_prediction_tables["ML"]["Actual_E"],

    all_prediction_tables["FULL"]["Actual_E"]

])

all_predicted = pd.concat([

    all_prediction_tables["QM"]["Predicted_E"],

    all_prediction_tables["ML"]["Predicted_E"],

    all_prediction_tables["FULL"]["Predicted_E"]

])

min_val = min(
    all_actual.min(),
    all_predicted.min()
)

max_val = max(
    all_actual.max(),
    all_predicted.max()
)

plt.plot(

    [min_val,max_val],

    [min_val,max_val],

    "k--",

    linewidth=2

)

plt.xlabel(
    "Actual Interaction Energy"
)

plt.ylabel(
    "Predicted Interaction Energy"
)

plt.title(
    "Descriptor Set Comparison"
)

plt.legend()

plt.tight_layout()

plt.show()

In [ ]:
plot_actual_vs_predicted(
    all_prediction_tables["QM"],
    "QM",
    "red"
)

plot_actual_vs_predicted(
    all_prediction_tables["ML"],
    "ML",
    "blue"
)

plot_actual_vs_predicted(
    all_prediction_tables["FULL"],
    "FULL",
    "green"
)

In [ ]:
# =========================================================
# ACTUAL vs PREDICTED PLOT
# =========================================================

def plot_actual_vs_predicted(
    prediction_df,
    descriptor_name,
    color
):

    # --------------------------------------------------
    # FONT SETTINGS
    # --------------------------------------------------

    plt.rcParams["font.family"] = "Times New Roman"
    plt.rcParams["font.size"] = 12
    plt.rcParams["font.weight"] = "bold"

    actual = prediction_df["Actual_E"]
    predicted = prediction_df["Predicted_E"]

    r2 = r2_score(
        actual,
        predicted
    )

    mae = mean_absolute_error(
        actual,
        predicted
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual,
            predicted
        )
    )

    # --------------------------------------------------
    # CREATE FIGURE
    # --------------------------------------------------

    plt.figure(
        figsize=(7,7)
    )

    plt.scatter(

        actual,

        predicted,

        color=color,

        s=90,

        alpha=0.85,

        edgecolors="black",

        linewidth=1.5

    )

    # --------------------------------------------------
    # DIAGONAL REFERENCE LINE
    # --------------------------------------------------

    min_val = min(
        actual.min(),
        predicted.min()
    )

    max_val = max(
        actual.max(),
        predicted.max()
    )

    plt.plot(

        [min_val, max_val],

        [min_val, max_val],

        linestyle="--",

        color="black",

        linewidth=2.5

    )

    # --------------------------------------------------
    # AXIS LABELS
    # --------------------------------------------------

    plt.xlabel(

        "Actual Interaction Energy (kcal/mol)",

        fontsize=12,

        fontweight="bold"

    )

    plt.ylabel(

        "Predicted Interaction Energy (kcal/mol)",

        fontsize=12,

        fontweight="bold"

    )

    plt.title(

        f"{descriptor_name} Validation",

        fontsize=12,

        fontweight="bold"

    )

    # --------------------------------------------------
    # METRICS
    # --------------------------------------------------

    plt.text(

        0.05,

        0.95,

        f"R$^2$ = {r2:.4f}\n"
        f"MAE = {mae:.3f}\n"
        f"RMSE = {rmse:.3f}",

        transform=plt.gca().transAxes,

        fontsize=12,

        fontweight="bold",

        verticalalignment="top"

    )

    # --------------------------------------------------
    # AXES STYLE
    # --------------------------------------------------

    ax = plt.gca()

    ax.set_xlim(min_val, max_val)
    ax.set_ylim(min_val, max_val)

    ax.tick_params(

        axis="both",

        direction="in",

        length=6,

        width=2,

        labelsize=12

    )

    for tick in ax.get_xticklabels():
        tick.set_fontweight("bold")

    for tick in ax.get_yticklabels():
        tick.set_fontweight("bold")

    for spine in ax.spines.values():

        spine.set_linewidth(2)

    # --------------------------------------------------
    # SAVE
    # --------------------------------------------------

    plt.tight_layout()

    plt.savefig(

        f"{descriptor_name}_Actual_vs_Predicted.png",

        dpi=600,

        bbox_inches="tight"

    )

    print(
        f"Saved: {descriptor_name}_Actual_vs_Predicted.png"
    )

    plt.show()

In [ ]:
plot_actual_vs_predicted(
    all_prediction_tables["QM"],
    "QM",
    "red"
)

plot_actual_vs_predicted(
    all_prediction_tables["ML"],
    "ML",
    "blue"
)

plot_actual_vs_predicted(
    all_prediction_tables["FULL"],
    "FULL",
    "green"
)

In [ ]:
# =========================================================
# ML SHAP ANALYSIS
# =========================================================

X_ml = df[ml_columns]

y_ml = df["E"]

scaler_ml = StandardScaler()

X_ml_scaled = pd.DataFrame(

    scaler_ml.fit_transform(
        X_ml
    ),

    columns=ml_columns
)

ml_model = XGBRegressor(
    **xgb_params
)

ml_model.fit(
    X_ml_scaled,
    y_ml
)

In [ ]:
# =========================================================
# CALCULATE SHAP VALUES
# =========================================================

explainer_ml = shap.TreeExplainer(
    ml_model
)

shap_values_ml = (
    explainer_ml.shap_values(
        X_ml_scaled
    )
)

In [ ]:
# =========================================================
# SHAP SUMMARY PLOT
# =========================================================

shap.summary_plot(

    shap_values_ml,

    X_ml_scaled,

    show=False

)

plt.tight_layout()

plt.savefig(

    "ML_SHAP_Summary.png",

    dpi=600,

    bbox_inches="tight"

)

print(
    "Saved: ML_SHAP_Summary.png"
)

plt.show()

In [ ]:
# =========================================================
# SHAP TABLE
# =========================================================

ml_shap_table = pd.DataFrame({

    "Feature":
        X_ml_scaled.columns,

    "Mean_Abs_SHAP":
        np.abs(
            shap_values_ml
        ).mean(axis=0)

})

ml_shap_table = (

    ml_shap_table

    .sort_values(

        "Mean_Abs_SHAP",

        ascending=False

    )

    .reset_index(drop=True)

)

ml_shap_table.to_csv(

    "ML_SHAP_Table.csv",

    index=False

)

print(
    "Saved: ML_SHAP_Table.csv"
)

ml_shap_table

In [ ]:
# =========================================================
# FULL SHAP ANALYSIS
# =========================================================

X_full = df[full_columns]

y_full = df["E"]

scaler_full = StandardScaler()

X_full_scaled = pd.DataFrame(

    scaler_full.fit_transform(
        X_full
    ),

    columns=full_columns
)

full_model = XGBRegressor(
    **xgb_params
)

full_model.fit(
    X_full_scaled,
    y_full
)

In [ ]:
# =========================================================
# CALCULATE SHAP VALUES
# =========================================================

explainer_full = shap.TreeExplainer(
    full_model
)

shap_values_full = (

    explainer_full.shap_values(
        X_full_scaled
    )

)

In [ ]:
# =========================================================
# SHAP SUMMARY PLOT
# =========================================================

shap.summary_plot(

    shap_values_full,

    X_full_scaled,

    show=False

)

plt.tight_layout()

plt.savefig(

    "FULL_SHAP_Summary.png",

    dpi=600,

    bbox_inches="tight"

)

print(
    "Saved: FULL_SHAP_Summary.png"
)

plt.show()

In [ ]:
# =========================================================
# SHAP TABLE
# =========================================================

full_shap_table = pd.DataFrame({

    "Feature":
        X_full_scaled.columns,

    "Mean_Abs_SHAP":
        np.abs(
            shap_values_full
        ).mean(axis=0)

})

full_shap_table = (

    full_shap_table

    .sort_values(

        "Mean_Abs_SHAP",

        ascending=False

    )

    .reset_index(drop=True)

)

full_shap_table.to_csv(

    "FULL_SHAP_Table.csv",

    index=False

)

print(
    "Saved: FULL_SHAP_Table.csv"
)

full_shap_table